# Call tools with LLMs

## Step 1: Specify tool definitions

We will specify the `calculator` function as a tool, so it can be used to perform calculations.

In [97]:
calculator_tool_definition = {
    "type": "function",
    "function": {
        "name": "calculator",
        "description": "Perform basic arithmetic operations.",
        "parameters": {
            "type": "object",
            "properties": {
                "operator": {
                    "type": "string",
                    "description": "Arithmetic operation to perform",
                    "enum": ["add", "subtract", "multiply", "divide"],                    
                },
                "first_number": {
                    "type": "number",
                    "description": "First number for the calculation"
                },
                "second_number": {
                    "type": "number",
                    "description": "Second number for the calculation"
                }
            },
            "required": ["operator", "first_number", "second_number"]
        }
    }
}

## Step 2: Implement the tool

Now, we define and implement the actual function that acts as a tool for our agent.

In [98]:
def calculator(operator: str, first_number: float, second_number: float):
    match operator:
        case "add":
            return first_number + second_number
        case "subtract":
            return first_number - second_number
        case "multiply":
            return first_number * second_number
        case "divide":
            return first_number / second_number
        case _:
            raise ValueError(f"Unsupported operator: {operator}")

In [99]:
assert 5 == calculator("add", 3, 2)
assert 1 == calculator("subtract", 3, 2)
assert 6 == calculator("multiply", 3, 2)
assert 1.5 == calculator("divide", 3, 2)

## Step 3: Executing the Tool Calling 

Given the tool definitions, the LLM decides which tool is needed. We'll see two examples with different responses. We configure a completion that knows about our calculator tool, and see if it uses it to answer different questions.

In [ ]:
from litellm import completion
tools = [calculator_tool_definition]

def ask_llm(question: str, history):
    history.append({"role": "user", "content": question})
    response = completion(
        model='ollama/gpt-oss:120b-cloud',
        messages=history,
        tools=tools)
  
    message = response.choices[0].message
    print(message.content)
    print(message.tool_calls)
    return message

In [85]:
ask_llm("What is the capital of South Korea?")

None
[ChatCompletionMessageToolCall(function=Function(arguments='{"query": "What is the capital of South Korea?"}', name='response_generator'), id='call_c5320f21-4485-4d27-97a2-a5fc3124a025', type='function')]


Message(content=None, role='assistant', tool_calls=[ChatCompletionMessageToolCall(function=Function(arguments='{"query": "What is the capital of South Korea?"}', name='response_generator'), id='call_c5320f21-4485-4d27-97a2-a5fc3124a025', type='function')], function_call=None, provider_specific_fields=None)

In [100]:
history = []
ai_message = ask_llm("What is 1234 x 5678?", history)

None
[ChatCompletionMessageToolCall(function=Function(arguments='{"operator": "multiply", "first_number": 1234, "second_number": 5678}', name='calculator'), id='call_4aa7755a-bb78-471d-b0b0-e3203a70769e', type='function')]


## Step 4: Running the tool

We extract the information needed to execute the requested commands.

In [89]:
import json

def call_tools(message, history):
    if message.tool_calls:
        for tool_call in message.tool_calls:
            print(tool_call)
            function_name = tool_call.function.name
            function_args = json.loads(tool_call.function.arguments)

            if function_name == "calculator":
                result = calculator(**function_args)

                history.append({
                    "role": "tool",
                    "tool_call_id": tool_call.id,
                    "content": str(result)
                })

In [ ]:
history.append(({
  "role": "assistant",
  "content": ai_message.content,
  "tool_calls": ai_message.tool_calls
}))

call_tools(ai_message, history)

final_response = completion(
    model="ollama/gpt-oss:120b-cloud",
    messages=history
)
print("Messages: ", history)
print("Final Answer: ", final_response.choices[0].message.content)



ChatCompletionMessageToolCall(function=Function(arguments='{"operator": "multiply", "first_number": 1234, "second_number": 5678}', name='calculator'), id='call_4aa7755a-bb78-471d-b0b0-e3203a70769e', type='function')
Messages:  [{'role': 'user', 'content': 'What is 1234 x 5678?'}, {'role': 'assistant', 'content': None, 'tool_calls': [ChatCompletionMessageToolCall(function=Function(arguments='{"operator": "multiply", "first_number": 1234, "second_number": 5678}', name='calculator'), id='call_4aa7755a-bb78-471d-b0b0-e3203a70769e', type='function')]}, {'role': 'tool', 'tool_call_id': 'call_4aa7755a-bb78-471d-b0b0-e3203a70769e', 'content': '7006652'}]
Final Answer:  ### Assistant:
Tool Calls: [
  {
    "id": "call_9d489b6c-b9d1-4c64-a36b-577c9a9c38d8",
    "type": "function",
    "function": {
      "name": "calculator",
      "arguments": {
        "operator": "divide",
        "first_number": 7006652,
        "second_number": 100
      }
    }
  }
]


/home/thomi/Lernen/Python/ai-agent-from-scratch/.venv/lib/python3.12/site-packages/pydantic/main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected 10 fields but got 6: Expected `Message` - serialized value may not be as expected [field_name='message', input_value=Message(content='### Assi... reasoning_content=None), input_type=Message])
  PydanticSerializationUnexpectedValue(Expected `StreamingChoices` - serialized value may not be as expected [field_name='choices', input_value=Choices(finish_reason='st...reasoning_content=None)), input_type=Choices])
  return self.__pydantic_serializer__.to_python(
